In [1]:
import subprocess, pandas as pd
from idd_tools.jobmon import collect_sacct, audit_run, EfficiencyBins

# Pull yesterday's jobs for your user
sacct_df = collect_sacct(user="bcreiner", start="2026-05-20", end="2026-05-21")

yesterday = sacct_df[sacct_df["start"].str.startswith("2026-05-20") & (sacct_df["state"] != "RUNNING")]

yesterday = yesterday[yesterday["state"] == "COMPLETED"].copy()
yesterday["mem_efficiency"] = yesterday["max_rss_gib"] / yesterday["req_mem_gib"]
yesterday["runtime_efficiency"] = yesterday["elapsed_seconds"] / (yesterday["timelimit_minutes"] * 60)
yesterday["template"] = yesterday["job_name"].str.replace(r"-\d+$", "", regex=True)
print(yesterday.groupby("template")[["req_mem_gib", "max_rss_gib"]].mean())
yesterday.groupby("template")[["mem_efficiency", "runtime_efficiency"]].describe()


                      req_mem_gib  max_rss_gib
template                                      
aggregate_finalize            2.0     1.998417
fit_malaria_pfpr_big         32.0     5.394302
interactive                  25.5     0.000000
predict_year_bin              1.0     0.556110


mem_efficiency                                          \
                              count      mean       std       min       25%   
template                                                                      
aggregate_finalize              1.0  0.999208       NaN  0.999208  0.999208   
fit_malaria_pfpr_big          863.0  0.168572  0.012413  0.131999  0.159569   
interactive                     2.0  0.000000  0.000000  0.000000  0.000000   
predict_year_bin             5655.0  0.556110  0.059846  0.291264  0.520353   

                                                   runtime_efficiency  \
                           50%       75%       max              count   
template                                                                
aggregate_finalize    0.999208  0.999208  0.999208                1.0   
fit_malaria_pfpr_big  0.166003  0.179039  0.195596              863.0   
interactive           0.000000  0.000000  0.000000                2.0   
predict_year_bin      0.555614  0.589643  0.851383             5655.0   

                                                                        \
                          mean       std       min       25%       50%   
template                                                                 
aggregate_finalize    0.245556       NaN  0.245556  0.245556  0.245556   
fit_malaria_pfpr_big  0.426299  0.266196  0.036597  0.189028  0.405000   
interactive           0.028435  0.039157  0.000747  0.014591  0.028435   
predict_year_bin      0.384231  0.075236  0.241667  0.310000  0.421667   

                                          
                           75%       max  
template                                  
aggregate_finalize    0.245556  0.245556  
fit_malaria_pfpr_big  0.629271  0.996389  
interactive           0.042279  0.056123  
predict_year_bin      0.445000  0.638333

In [2]:
sacct_df

,job_id,job_name,state,elapsed_seconds,max_rss_gib,req_mem_gib,timelimit_minutes,exit_code,start,end,raw_max_rss,raw_req_mem
0,15953645_54,scenario_template-4900,RUNNING,18360764,0.000000,10.0,60.0,0:0,2025-10-21T22:11:08,Unknown,,10G
1,15953645_55,scenario_template-4900,RUNNING,18360764,0.000000,10.0,60.0,0:0,2025-10-21T22:11:08,Unknown,,10G
2,15953645_67,scenario_template-4900,RUNNING,18360764,0.000000,10.0,60.0,0:0,2025-10-21T22:11:08,Unknown,,10G
3,15953645_68,scenario_template-4900,RUNNING,18360764,0.000000,10.0,60.0,0:0,2025-10-21T22:11:08,Unknown,,10G
4,31494920_3,create_climada_input_data-10,RUNNING,15866367,0.000000,40.0,60.0,0:0,2025-11-19T18:04:25,Unknown,,40G
...,...,...,...,...,...,...,...,...,...,...,...,...
8117,42464237_1,predict_year_bin-3326,COMPLETED,176,0.523575,1.0,10.0,0:0,2026-05-20T18:50:04,2026-05-20T18:53:00,549008K,1G
8118,42464238_1,predict_year_bin-4011,COMPLETED,181,0.482658,1.0,10.0,0:0,2026-05-20T18:50:04,2026-05-20T18:53:05,506104K,1G
8119,42464239_1,predict_year_bin-2624,COMPLETED,176,0.573647,1.0,10.0,0:0,2026-05-20T18:50:04,2026-05-20T18:53:00,601512K,1G
8120,42467014_1,aggregate_finalize-1,OUT_OF_MEMORY,185,0.998409,1.0,10.0,0:125,2026-05-20T18:55:43,2026-05-20T18:58:48,1046908K,1G


In [3]:
from jobmon.client.api import workflow_tasks, task_status

tasks = workflow_tasks(579099, limit=100000)
details = task_status(tasks["TASK_ID"].tolist())
# details has DISTRIBUTOR_ID (slurm job IDs) + RESOURCE_USAGE already
job_ids = details["DISTRIBUTOR_ID"].dropna().tolist()
sacct_df = collect_sacct(jobids=job_ids)

2026-05-22 10:23:56 [debug    ] workflow id: 579099           
2026-05-22 10:23:57 [debug    ] Making HTTP request            request_type=get route=https://jobmon.scicomp.ihme.washington.edu/api/v3/workflow/579099/workflow_tasks
2026-05-22 10:23:57 [debug    ] task_status task_ids:[345877246, 345877245, 345877244, 345877243, 345877242, 345877241, 345877240, 345877239, 345877238, 345877237, 345877236, 345877235, 345877234, 345877233, 345877232, 345877231, 345877230, 345877229, 345877228, 345877227, 345877226, 345877225, 345877224, 345877223, 345877222, 345877221, 345877220, 345877219, 345877218, 345877217, 345877216, 345877215, 345877214, 345877213, 345877212, 345877211, 345877210, 345877209, 345877208, 345877207, 345877206, 345877205, 345877204, 345877203, 345877202, 345877201, 345877200, 345877199, 345877198, 345877197, 345877196, 345877195, 345877194, 345877193, 345877192, 345877191, 345877190, 345877189, 345877188, 345877187, 345877186, 345877185, 345877184, 345877183, 345877182, 3

In [4]:
sacct_df

,job_id,job_name,state,elapsed_seconds,max_rss_gib,req_mem_gib,timelimit_minutes,exit_code,start,end,raw_max_rss,raw_req_mem
0,42450090_1,predict_year_bin-3509,COMPLETED,295,0.558594,1.0,10.0,0:0,2026-05-20T17:57:49,2026-05-20T18:02:44,572M,1G
1,42454167_1,predict_year_bin-3785,COMPLETED,182,0.534428,1.0,10.0,0:0,2026-05-20T18:25:33,2026-05-20T18:28:35,560388K,1G
2,42454168_1,predict_year_bin-4197,COMPLETED,197,0.550732,1.0,10.0,0:0,2026-05-20T18:25:33,2026-05-20T18:28:50,577484K,1G
3,42453530_1,predict_year_bin-4718,COMPLETED,273,0.553284,1.0,10.0,0:0,2026-05-20T18:21:10,2026-05-20T18:25:43,580160K,1G
4,42455351_1,predict_year_bin-5301,COMPLETED,262,0.592243,1.0,10.0,0:0,2026-05-20T18:28:50,2026-05-20T18:33:12,621012K,1G
...,...,...,...,...,...,...,...,...,...,...,...,...
2202,42464233_1,predict_year_bin-5641,COMPLETED,166,0.511459,1.0,10.0,0:0,2026-05-20T18:50:01,2026-05-20T18:52:47,536304K,1G
2203,42464234_1,predict_year_bin-4385,COMPLETED,169,0.507366,1.0,10.0,0:0,2026-05-20T18:50:01,2026-05-20T18:52:50,532012K,1G
2204,42464238_1,predict_year_bin-4011,COMPLETED,181,0.482658,1.0,10.0,0:0,2026-05-20T18:50:04,2026-05-20T18:53:05,506104K,1G
2205,42467014_1,aggregate_finalize-1,OUT_OF_MEMORY,185,0.998409,1.0,10.0,0:125,2026-05-20T18:55:43,2026-05-20T18:58:48,1046908K,1G


# Post-2000 pipeline resource audit
Covers two sacct windows:
- **2026-05-17**: fit (`fit_component`, 31 590 tasks) + evaluate (`evaluate_worker_taskfile`, 1 787 tasks)
- **2026-05-20**: predict workflow 579099 (`predict_year_bin`, 5 655 tasks; `aggregate_finalize`, 2 tasks)

Goal: calibrate allocation targets for the new-data rerun.

In [ ]:
import numpy as np
from idd_tools.jobmon import collect_sacct

PIPELINE_TEMPLATES = {
    "fit":      ["fit_component"],
    "evaluate": ["evaluate_worker_taskfile"],
    "predict":  ["predict_year_bin", "aggregate_finalize",
                 "aggregate_basin", "aggregate_scenario",
                 "aggregate_storm_draw", "aggregate_year_bin"],
}
ALL_PIPELINE = [t for ts in PIPELINE_TEMPLATES.values() for t in ts]

def resource_summary(df: pd.DataFrame) -> pd.DataFrame:
    df = df[df["state"] == "COMPLETED"].copy()
    df["template"] = df["job_name"].str.replace(r"-\d+$", "", regex=True)
    df = df[df["template"].isin(ALL_PIPELINE)]
    return (
        df.groupby("template")
        .agg(
            n            = ("job_id", "count"),
            req_mem_gib  = ("req_mem_gib", "first"),
            mean_mem_gib = ("max_rss_gib", "mean"),
            med_mem_gib  = ("max_rss_gib", "median"),
            max_mem_gib  = ("max_rss_gib", "max"),
            timelimit_m  = ("timelimit_minutes", "first"),
            mean_rt_s    = ("elapsed_seconds", "mean"),
            med_rt_s     = ("elapsed_seconds", "median"),
            max_rt_s     = ("elapsed_seconds", "max"),
        )
        .assign(
            mem_eff = lambda d: (d["mean_mem_gib"] / d["req_mem_gib"]).round(3),
            rt_eff  = lambda d: (d["mean_rt_s"] / (d["timelimit_m"] * 60)).round(3),
        )
        .round(3)
        .reset_index()
    )

# --- 2026-05-17: fit + evaluate ---
raw17 = collect_sacct(user="bcreiner", start="2026-05-17", end="2026-05-18")
stats17 = resource_summary(raw17)

# --- 2026-05-20: predict (wf 579099) — reuse sacct_df from cell above ---
stats20 = resource_summary(sacct_df)   # sacct_df = wf-579099 job-id pull

print("=== 2026-05-17  fit + evaluate ===")
display(stats17)
print("\n=== 2026-05-20  predict wf 579099 ===")
display(stats20)

In [ ]:
# --- Flags: over-provisioned (mem_eff < 0.30) or wall-close (rt_eff > 0.80) ---
combined = pd.concat([stats17, stats20], ignore_index=True).drop_duplicates("template")

print("Over-provisioned  (mean_mem / req_mem < 0.30):")
display(combined[combined["mem_eff"] < 0.30][["template","n","req_mem_gib","mean_mem_gib","max_mem_gib","mem_eff"]])

print("\nWall-close  (mean_runtime / timelimit > 0.80):")
wall_close = combined[combined["rt_eff"] > 0.80]
if wall_close.empty:
    print("  none")
else:
    display(wall_close[["template","n","timelimit_m","mean_rt_s","max_rt_s","rt_eff"]])

print("\naggregate_finalize OOM detail (first attempt at 1G OOM'd; second at 2G used 99.9%):")
raw20 = collect_sacct(user="bcreiner", start="2026-05-20", end="2026-05-21")
raw20["template"] = raw20["job_name"].str.replace(r"-\d+$", "", regex=True)
display(raw20[raw20["template"] == "aggregate_finalize"][
    ["job_id","state","elapsed_seconds","max_rss_gib","req_mem_gib","timelimit_minutes","start","end"]
])

In [ ]:
# --- predict_year_bin bimodal runtime analysis ---
import matplotlib.pyplot as plt

pyb = raw20[(raw20["template"] == "predict_year_bin") & (raw20["state"] == "COMPLETED")].copy()
pyb["start_dt"] = pd.to_datetime(pyb["start"])
pyb["start_min"] = (pyb["start_dt"] - pyb["start_dt"].min()).dt.total_seconds() / 60
pyb["mode"] = pd.cut(pyb["elapsed_seconds"], bins=[0, 225, 9999],
                     labels=["fast (<225s)", "slow (≥225s)"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: runtime histogram
axes[0].hist(pyb["elapsed_seconds"], bins=range(120, 420, 15), edgecolor="white", linewidth=0.3)
axes[0].axvline(225, color="red", linestyle="--", linewidth=1, label="mode split (225s)")
axes[0].set_xlabel("elapsed_seconds"); axes[0].set_ylabel("task count")
axes[0].set_title("predict_year_bin runtime distribution (wf 579099)")
axes[0].legend()

# Right: runtime vs start_time coloured by mode
for label, grp in pyb.groupby("mode"):
    axes[1].scatter(grp["start_min"], grp["elapsed_seconds"],
                    s=1, alpha=0.3, label=label)
axes[1].set_xlabel("minutes after first task"); axes[1].set_ylabel("elapsed_seconds")
axes[1].set_title("Runtime vs start time — both modes span full window")
axes[1].legend(markerscale=6)

plt.tight_layout()
plt.show()

print(f"\nMode counts:  {pyb['mode'].value_counts().to_dict()}")
print("\nStart time by mode (minutes after first task):")
display(pyb.groupby("mode")["start_min"].describe().round(1))

## Allocation targets for the new-data rerun

| Template | Current | Observed max RSS | Observed max runtime | **New target** | Note |
|---|---|---|---|---|---|
| `fit_component` | 1G / 5m | 0.28G | 100s | **512M / 5m** | over-provisioned 4× on mem |
| `evaluate_worker_taskfile` | 2G / 30m | 0.58G | 222s | **1G / 5m** | over-provisioned 7× mem, 8× time |
| `predict_year_bin` | 1G / 10m | 0.85G | 383s | **1G / 10m** | leave as-is — well sized |
| `aggregate_finalize` | 2G / 15m | 2.00G (99.9%) | 221s | **3G / 15m** | ⚠️ OOM risk with new data |

**Bimodal predict_year_bin:** fast mode (~180s, 2 587 tasks) and slow mode (~255s, 3 068 tasks) are **temporally interleaved** — both span the full 80-minute workflow window with heavily overlapping job-ID ranges.  Conclusion: two work-size classes (likely years with different TC event counts), not a contention artifact.  No resource concern; max runtime 383s is well within 10m.